In [ ]:
# @title Simulation Systemic Risk - MDPI Publication Quality -*- coding: utf-8 -*-

from itertools import permutations
import matplotlib.pyplot as plt
from time import process_time as ptime
import numpy as np
import pandas as pd

# ➕➕➕ IMPORTS FOR PUBLICATION-QUALITY FIGURES ➕➕➕
import seaborn as sns
import matplotlib as mpl
from matplotlib import rcParams

# ============================================================================
# PUBLICATION STYLE CONFIGURATION (LaTeX Beamer Academic Presentation Style)
# ============================================================================

# Font Configuration - LaTeX Academic Style
rcParams['font.family'] = 'serif'
rcParams['font.serif'] = ['Computer Modern Roman', 'Times New Roman', 'DejaVu Serif']
rcParams['font.size'] = 10
rcParams['axes.labelsize'] = 12
rcParams['axes.titlesize'] = 13
rcParams['xtick.labelsize'] = 10
rcParams['ytick.labelsize'] = 10
rcParams['legend.fontsize'] = 10
rcParams['figure.titlesize'] = 14
rcParams['text.usetex'] = False  # Set to True if LaTeX is installed
rcParams['mathtext.fontset'] = 'cm'  # Computer Modern math fonts

# Line and Border Configuration - Clean Professional Style
rcParams['axes.linewidth'] = 1.5
rcParams['grid.linewidth'] = 0.8
rcParams['lines.linewidth'] = 2.0
rcParams['patch.linewidth'] = 1.5

# Color Configuration - All text BLACK
rcParams['text.color'] = 'black'
rcParams['axes.labelcolor'] = 'black'
rcParams['xtick.color'] = 'black'
rcParams['ytick.color'] = 'black'
rcParams['axes.edgecolor'] = 'black'

# Grid Configuration - Subtle grey
rcParams['grid.color'] = '#CCCCCC'
rcParams['grid.alpha'] = 0.3
rcParams['grid.linestyle'] = '--'

# Figure Configuration - High quality for MDPI
rcParams['figure.dpi'] = 600  # MDPI requirement: ≥600 dpi
rcParams['savefig.dpi'] = 600
rcParams['savefig.bbox'] = 'tight'
rcParams['savefig.facecolor'] = 'white'
rcParams['savefig.edgecolor'] = 'none'
rcParams['savefig.format'] = 'png'

# LaTeX Beamer Color Palette (Academic Presentation Style)
# Based on classic beamer themes: blues, greys, professional
colors = {
    'beamer_blue': '#003366',      # Dark navy blue (primary)
    'beamer_light_blue': '#4682B4', # Steel blue (secondary)
    'beamer_grey': '#707070',       # Medium grey
    'beamer_light_grey': '#B0B0B0', # Light grey
    'beamer_dark_grey': '#404040',  # Dark grey
    'accent_blue': '#5F84A2',       # Accent blue-grey
    'background': '#F5F5F5',        # Off-white background
    'grid': '#CCCCCC',              # Grid color
    'text': '#000000'               # All text: BLACK
}

# Corporate Finance Color Palette (Professional Gradient)
# For bar charts and categorical data
corporate_palette = {
    'stable': '#E0E7ED',      # Very light blue-grey (low risk)
    'low': '#A8BDCC',         # Light blue-grey
    'medium': '#5F84A2',      # Medium slate blue
    'elevated': '#2E5A7D',    # Deep blue-grey
    'critical': '#1A3A52'     # Dark navy (high risk)
}

# ============================================================================
# HELPER FUNCTION: EXTRACT QUARTER RESULTS
# ============================================================================
# This function extracts the summary statistics for the network

def extract_quarter_results(E, H, PH, count, quarter_name):
    """
    Extract all statistics for one quarter

    Parameters:
    -----------
    E : list
        Equity losses for all scenarios
    H : list
        External debt losses for all scenarios
    PH : list
        Bank failure counts [P(0), P(1), P(2), P(3), P(4)]
    count : int
        Total number of scenarios
    quarter_name : str
        Quarter label (e.g., 'Q1 2009')

    Returns:
    --------
    dict
        Dictionary containing all summary statistics
    """
    E_arr = np.array(E)
    H_arr = np.array(H)
    SR_arr = E_arr + H_arr

    return {
        'Quarter': quarter_name,
        'Mean_E': np.mean(E_arr),
        'Min_E': np.min(E_arr),
        'Max_E': np.max(E_arr),
        'Mean_H': np.mean(H_arr),
        'Min_H': np.min(H_arr),
        'Max_H': np.max(H_arr),
        'Mean_SR': np.mean(SR_arr),
        'Min_SR': np.min(SR_arr),
        'Max_SR': np.max(SR_arr),
        'Std_SR': np.std(SR_arr),
        'P_0_banks': PH[0] / count,
        'P_1_bank': PH[1] / count,
        'P_2_banks': PH[2] / count,
        'P_3_banks': PH[3] / count,
        'P_4_banks': PH[4] / count,
        'P_at_least_1': 1 - (PH[0] / count),
        'P_at_least_2': 1 - (PH[0] + PH[1]) / count,
        'Expected_failures': sum(i * PH[i] / count for i in range(5)),
        'Equity_share': np.mean(E_arr) / np.mean(SR_arr) if np.mean(SR_arr) > 0 else 0,
        'Ext_debt_share': np.mean(H_arr) / np.mean(SR_arr) if np.mean(SR_arr) > 0 else 0,
        'N_scenarios': count
    }

# ============================================================================
# PUBLICATION FIGURE CREATION FUNCTION (LaTeX Beamer Style)
# ============================================================================

def create_publication_figures(E, H, PH, count, quarter_label, quarter_name):
    """
    Create publication-quality figures in LaTeX Beamer academic style

    MDPI Requirements:
    - Resolution: 600+ dpi ✓
    - Format: PNG, TIFF ✓
    - Color: RGB 8-bit ✓
    - Fonts: All BLACK ✓
    - Style: Professional academic ✓

    Parameters:
    -----------
    E : list
        Equity losses (millions USD)
    H : list
        External debt losses (millions USD)
    PH : list
        Bank failure probability histogram
    count : int
        Total scenarios
    quarter_label : str
        File name label (e.g., 'Q1_2009')
    quarter_name : str
        Display name (e.g., 'Q1 2009')
    """

    # Convert to numpy arrays (values in MILLIONS USD)
    E_arr = np.array(E)
    H_arr = np.array(H)
    SR_arr = E_arr + H_arr

    print(f"\n{'='*80}")
    print(f"Creating publication figures for {quarter_name}")
    print(f"{'='*80}")
    print(f"Equity (E):        min=${min(E_arr):>12,.0f}M  max=${max(E_arr):>12,.0f}M  mean=${np.mean(E_arr):>12,.0f}M")
    print(f"Ext. Debt (H):     min=${min(H_arr):>12,.0f}M  max=${max(H_arr):>12,.0f}M  mean=${np.mean(H_arr):>12,.0f}M")
    print(f"Systemic Risk (SR): min=${min(SR_arr):>12,.0f}M  max=${max(SR_arr):>12,.0f}M  mean=${np.mean(SR_arr):>12,.0f}M")
    print(f"Scenarios: {count:,}")

    # ========================================================================
    # FIGURE 1: EQUITY LOSS DISTRIBUTION (Grey-Blue Academic Style)
    # ========================================================================

    fig = plt.figure(figsize=(8, 6), dpi=600, facecolor='white')
    ax = fig.add_subplot(111)

    # Histogram with professional blue-grey color
    counts, bins, patches = ax.hist(E_arr, bins=50, density=True,
                                     color=colors['beamer_light_blue'],
                                     alpha=0.75,
                                     edgecolor='black', linewidth=1.0)

    # All labels in BLACK
    ax.set_xlabel('Equity Loss (Millions USD)', fontsize=12, fontweight='bold', color='black')
    ax.set_ylabel('Probability Density', fontsize=12, fontweight='bold', color='black')
    ax.set_title(f'Distribution of Equity Losses\n{quarter_name}',
                 fontsize=14, fontweight='bold', color='black', pad=15)

    # Grid - subtle grey
    ax.grid(True, alpha=0.3, linestyle='--', color='#CCCCCC', linewidth=0.8)
    ax.set_axisbelow(True)

    # Mean line - dark blue
    mean_E = np.mean(E_arr)
    ax.axvline(mean_E, color=colors['beamer_blue'], linestyle='--', linewidth=2.5,
               label=f'Mean: ${mean_E:,.0f}M', alpha=0.9)

    # Legend with BLACK text
    legend = ax.legend(loc='upper right', fontsize=10, frameon=True,
                      shadow=True, fancybox=True)
    plt.setp(legend.get_texts(), color='black')

    # Tick parameters - BLACK
    ax.tick_params(axis='both', colors='black', labelsize=10)

    # Spine configuration - BLACK
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'Fig_Equity_{quarter_label}.png', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(f'Fig_Equity_{quarter_label}.tiff', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"  ✓ Saved Fig_Equity_{quarter_label}.png (600 dpi)")

    # ========================================================================
    # FIGURE 2: EXTERNAL DEBT LOSS DISTRIBUTION (Grey-Blue Academic Style)
    # ========================================================================

    fig = plt.figure(figsize=(8, 6), dpi=600, facecolor='white')
    ax = fig.add_subplot(111)

    # Histogram with professional grey color
    counts, bins, patches = ax.hist(H_arr, bins=50, density=True,
                                     color=colors['beamer_grey'],
                                     alpha=0.75,
                                     edgecolor='black', linewidth=1.0)

    # All labels in BLACK
    ax.set_xlabel('External Debt Loss (Millions USD)', fontsize=12, fontweight='bold', color='black')
    ax.set_ylabel('Probability Density', fontsize=12, fontweight='bold', color='black')
    ax.set_title(f'Distribution of External Debt Losses\n{quarter_name}',
                 fontsize=14, fontweight='bold', color='black', pad=15)

    # Grid - subtle grey
    ax.grid(True, alpha=0.3, linestyle='--', color='#CCCCCC', linewidth=0.8)
    ax.set_axisbelow(True)

    # Mean line - dark blue
    mean_H = np.mean(H_arr)
    ax.axvline(mean_H, color=colors['beamer_blue'], linestyle='--', linewidth=2.5,
               label=f'Mean: ${mean_H:,.0f}M', alpha=0.9)

    # Legend with BLACK text
    legend = ax.legend(loc='upper right', fontsize=10, frameon=True,
                      shadow=True, fancybox=True)
    plt.setp(legend.get_texts(), color='black')

    # Tick parameters - BLACK
    ax.tick_params(axis='both', colors='black', labelsize=10)

    # Spine configuration - BLACK
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'Fig_ExtDebt_{quarter_label}.png', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(f'Fig_ExtDebt_{quarter_label}.tiff', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"  ✓ Saved Fig_ExtDebt_{quarter_label}.png (600 dpi)")

    # ========================================================================
    # FIGURE 3: TOTAL SYSTEMIC RISK DISTRIBUTION (Dark Blue Academic Style)
    # ========================================================================

    fig = plt.figure(figsize=(8, 6), dpi=600, facecolor='white')
    ax = fig.add_subplot(111)

    # Histogram with dark blue color
    counts, bins, patches = ax.hist(SR_arr, bins=50, density=True,
                                     color=colors['beamer_blue'],
                                     alpha=0.70,
                                     edgecolor='black', linewidth=1.0)

    # All labels in BLACK
    ax.set_xlabel('Total Systemic Risk (Millions USD)', fontsize=12, fontweight='bold', color='black')
    ax.set_ylabel('Probability Density', fontsize=12, fontweight='bold', color='black')
    ax.set_title(f'Distribution of Total Systemic Risk (E+H)\n{quarter_name}',
                 fontsize=14, fontweight='bold', color='black', pad=15)

    # Grid - subtle grey
    ax.grid(True, alpha=0.3, linestyle='--', color='#CCCCCC', linewidth=0.8)
    ax.set_axisbelow(True)

    # Mean line - steel blue
    mean_SR = np.mean(SR_arr)
    ax.axvline(mean_SR, color=colors['beamer_light_blue'], linestyle='--', linewidth=2.5,
               label=f'Mean: ${mean_SR:,.0f}M', alpha=0.9)

    # Legend with BLACK text
    legend = ax.legend(loc='upper right', fontsize=10, frameon=True,
                      shadow=True, fancybox=True)
    plt.setp(legend.get_texts(), color='black')

    # Statistics text box - BLACK text on light background
    textstr = f'Mean: ${mean_SR:,.0f}M\nStd: ${np.std(SR_arr):,.0f}M\nN = {count:,}'
    props = dict(boxstyle='round,pad=0.5', facecolor='#E8EEF2',
                edgecolor='black', linewidth=1.5, alpha=0.9)
    ax.text(0.68, 0.97, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=props, color='black', fontweight='bold')

    # Tick parameters - BLACK
    ax.tick_params(axis='both', colors='black', labelsize=10)

    # Spine configuration - BLACK
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'Fig_SystRisk_{quarter_label}.png', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(f'Fig_SystRisk_{quarter_label}.tiff', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"  ✓ Saved Fig_SystRisk_{quarter_label}.png (600 dpi)")

    # ========================================================================
    # FIGURE 4: BANK FAILURE PROBABILITY (Corporate Blue-Grey Gradient)
    # ========================================================================

    fig = plt.figure(figsize=(8, 6), dpi=600, facecolor='white')
    ax = fig.add_subplot(111)

    x_pos = np.arange(5)
    probs = np.array(PH) / count

    # Corporate blue-grey gradient palette (LaTeX Beamer style)
    # Light grey → Dark navy blue (increasing severity)
    colors_bars = [
        corporate_palette['stable'],    # 0 banks: Very light blue-grey
        corporate_palette['low'],        # 1 bank: Light blue-grey
        corporate_palette['medium'],     # 2 banks: Medium slate blue
        corporate_palette['elevated'],   # 3 banks: Deep blue-grey
        corporate_palette['critical']    # 4 banks: Dark navy (critical)
    ]

    bars = ax.bar(x_pos, probs, width=0.65, color=colors_bars,
                  alpha=0.90, edgecolor='black', linewidth=1.8)

    # Add percentage labels - BLACK text
    for i, (bar, prob) in enumerate(zip(bars, probs)):
        height = bar.get_height()
        if height > 0.001:  # Only show if visible
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{prob*100:.2f}%',
                   ha='center', va='bottom', fontsize=11,
                   fontweight='bold', color='black')

    # All labels in BLACK
    ax.set_xlabel('Number of Banks Failed', fontsize=12, fontweight='bold', color='black')
    ax.set_ylabel('Probability', fontsize=12, fontweight='bold', color='black')
    ax.set_title(f'Bank Failure Probability Distribution\n{quarter_name}',
                 fontsize=14, fontweight='bold', color='black', pad=15)

    # X-axis configuration
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['0', '1', '2', '3', '4'], fontsize=11, color='black')
    ax.set_ylim(0, max(probs) * 1.18)  # Add 18% space for labels

    # Format y-axis as percentage
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))

    # Grid - horizontal only, subtle grey
    ax.grid(True, alpha=0.3, linestyle='--', axis='y', color='#CCCCCC', linewidth=0.8)
    ax.set_axisbelow(True)

    # Tick parameters - BLACK
    ax.tick_params(axis='both', colors='black', labelsize=10)

    # Spine configuration - BLACK, minimal
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'Fig_Failures_{quarter_label}.png', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(f'Fig_Failures_{quarter_label}.tiff', dpi=600, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"  ✓ Saved Fig_Failures_{quarter_label}.png (600 dpi)")

    print(f"{'='*80}")
    print(f"Completed all figures for {quarter_name}")
    print(f"{'='*80}\n")

# ============================================================================
# CONFIRMATION MESSAGE
# ============================================================================

print("\n" + "="*80)
print("✓✓✓ PUBLICATION-QUALITY FIGURE CONFIGURATION LOADED ✓✓✓")
print("="*80)
print("\nConfiguration:")
print("  • Style: LaTeX Beamer Academic Presentation")
print("  • Resolution: 600 DPI (MDPI compliant)")
print("  • Fonts: All BLACK, serif family")
print("  • Colors: Professional blue-grey gradient")
print("  • Format: PNG + TIFF dual export")
print("  • Grid: Subtle grey, academic style")
print("\nColor Palette:")
print(f"  • Primary: Dark Navy Blue ({colors['beamer_blue']})")
print(f"  • Secondary: Steel Blue ({colors['beamer_light_blue']})")
print(f"  • Tertiary: Medium Grey ({colors['beamer_grey']})")
print(f"  • All Text: BLACK ({colors['text']})")
print("\nCorporate Gradient (Bar Charts):")
print(f"  • 0 banks (stable):   Light grey-blue ({corporate_palette['stable']})")
print(f"  • 1 bank (low):       Light blue-grey ({corporate_palette['low']})")
print(f"  • 2 banks (medium):   Slate blue ({corporate_palette['medium']})")
print(f"  • 3 banks (elevated): Deep blue-grey ({corporate_palette['elevated']})")
print(f"  • 4 banks (critical): Dark navy ({corporate_palette['critical']})")
print("\nReady for simulation runs!")
print("="*80 + "\n")

In [2]:
# @title Initialize results list
all_quarterly_results = []

In [ ]:
# @title Shock Distribution Class - FINAL PUBLICATION VERSION
import numpy as np

## This is the main class that creates the shock distribution 
class ShockDistribution:
    """
    Empirically-calibrated stochastic shock generator.

    Calibrated to verified external SIFI failure-scale proxies during 2007-2009.
    External failures preserve identification: shock is exogenous to the
    four-bank network being modeled (Citi, JPM, BofA, WFC).

    Calibration set (failure-scale proxies, $B):
    ┌─────────────────────┬───────────┬─────────────────────────────────────┐
    │ Institution         │ Magnitude │ Source                              │
    ├─────────────────────┼───────────┼─────────────────────────────────────┤
    │ Wachovia            │ $812B     │ Alvarez 2010, Fed/FCIC testimony    │
    │ Merrill Lynch       │ $875B     │ Federal Reserve approval order (2008)    │
    │ Lehman Brothers     │ $639B     │ Bankruptcy petition #08-13555       │
    │ Bear Stearns        │ $400B     │ Fed March 2008 (assets at rescue)   │
    │ Washington Mutual   │ $307B     │ FDIC Receivership Report            │
    │ AIG                 │ $85B      │ Fed Section 129 (initial facility)  │
    └─────────────────────┴───────────┴─────────────────────────────────────┘

    Note: Magnitudes are failure-scale proxies (assets at event or initial
    support), not realized loss accounting. AIG uses initial facility rather
    than total assets to reflect transmitted shock magnitude to banking sector.

    Reference: Blinder (2014) estimates median SIFI ~$442B. Our implied median
    ($389B) differs by -12%, which we treat as acceptable calibration error.
    """

    # Verified external failure-scale proxies ($ billions)
    HISTORICAL_CRISES = np.array([812, 875, 639, 400, 307, 85], dtype=np.float64)

    # VIX-based quarterly stress factors (Q3 2007 = baseline)
    # Source: S&P Capital IQ, CBOE Volatility S&P 500 Index (^VIX)
    # Methodology: stress_factor(q) = mean_VIX(q) / mean_VIX(Q3 2007)
    # Baseline: Q3 2007 mean VIX = 21.59
    STRESS_FACTORS = {
        'Q3 2007': 1.00,  # VIX avg = 21.59 (baseline)
        'Q4 2007': 1.02,  # VIX avg = 22.03
        'Q1 2008': 1.21,  # VIX avg = 26.12
        'Q2 2008': 0.96,  # VIX avg = 20.67
        'Q3 2008': 1.16,  # VIX avg = 25.07
        'Q4 2008': 2.71,  # VIX avg = 58.60
    }

    def __init__(self, distribution_type='lognormal', seed=42):
        if distribution_type not in ('lognormal', 'time_varying'):
            raise ValueError(
                f"Unknown distribution_type '{distribution_type}'. "
                "Valid options: 'lognormal', 'time_varying'"
            )

        if np.any(self.HISTORICAL_CRISES <= 0):
            raise ValueError("HISTORICAL_CRISES must contain positive values only")

        self.dist_type = distribution_type
        self.rng = np.random.default_rng(seed)

        # Fit log-normal parameters (ddof=0: full reference population)
        self.ln_mu = np.mean(np.log(self.HISTORICAL_CRISES))
        self.ln_sigma = np.std(np.log(self.HISTORICAL_CRISES), ddof=0)

        self.implied_median = np.exp(self.ln_mu)
        self.implied_mean = np.exp(self.ln_mu + self.ln_sigma**2 / 2)

        print(f"ShockDistribution initialized:")
        print(f"  Type: {distribution_type}")
        print(f"  Calibration: {len(self.HISTORICAL_CRISES)} external failure-scale proxies")
        print(f"  Log-normal params: μ={self.ln_mu:.4f}, σ={self.ln_sigma:.4f}")
        print(f"  Implied median: ${self.implied_median:.0f}B")
        print(f"  Implied mean: ${self.implied_mean:.0f}B")
        print(f"  Implied 95% range: [${np.exp(self.ln_mu - 1.96*self.ln_sigma):.0f}B, "
              f"${np.exp(self.ln_mu + 1.96*self.ln_sigma):.0f}B]")
        print(f"  Blinder (2014) reference: $442B (Δ = {(self.implied_median/442 - 1)*100:+.1f}%)")

    def sample(self, n_samples=1, quarter=None):
        """
        Generate shock samples from the calibrated distribution.

        Returns
        -------
        np.ndarray
            Shock values in MILLIONS USD (simulation compatibility)
        """
        if self.dist_type == 'lognormal':
            shocks_billions = self.rng.lognormal(
                mean=self.ln_mu,
                sigma=self.ln_sigma,
                size=n_samples
            )

        elif self.dist_type == 'time_varying':
            if quarter is None:
                raise ValueError(
                    "quarter parameter required for time_varying distribution. "
                    f"Valid quarters: {list(self.STRESS_FACTORS.keys())}"
                )

            if quarter not in self.STRESS_FACTORS:
                raise ValueError(
                    f"Unknown quarter '{quarter}'. "
                    f"Valid quarters: {list(self.STRESS_FACTORS.keys())}"
                )

            stress_factor = self.STRESS_FACTORS[quarter]

            # √stress scaling: μ_adj = μ + ½·ln(stress)
            # With σ fixed, median and mean scale by √stress.
            mu_adj = self.ln_mu + 0.5 * np.log(stress_factor)

            shocks_billions = self.rng.lognormal(
                mean=mu_adj,
                sigma=self.ln_sigma,
                size=n_samples
            )

        else:
            # Guard against corrupted state
            raise RuntimeError(f"Invalid dist_type state: {self.dist_type}")

        return shocks_billions * 1000

    def get_implied_stats(self, quarter=None):
        """
        Compute analytic implied statistics (no Monte Carlo noise).

        Parameters
        ----------
        quarter : str, optional
            Required for 'time_varying', ignored for 'lognormal'

        Returns
        -------
        dict
            Analytic median, mean, and 95% range in BILLIONS USD
        """
        # Enforce consistent semantics based on distribution type
        if self.dist_type == 'time_varying':
            if quarter is None:
                raise ValueError(
                    "quarter parameter required for time_varying distribution. "
                    f"Valid quarters: {list(self.STRESS_FACTORS.keys())}"
                )
            if quarter not in self.STRESS_FACTORS:
                raise ValueError(f"Unknown quarter '{quarter}'")
            stress_factor = self.STRESS_FACTORS[quarter]
            mu = self.ln_mu + 0.5 * np.log(stress_factor)

        elif self.dist_type == 'lognormal':
            # Ignore quarter for lognormal (baseline distribution)
            mu = self.ln_mu

        else:
            raise RuntimeError(f"Invalid dist_type state: {self.dist_type}")

        sigma = self.ln_sigma

        return {
            'median_B': np.exp(mu),
            'mean_B': np.exp(mu + sigma**2 / 2),
            'p2.5_B': np.exp(mu - 1.96 * sigma),
            'p97.5_B': np.exp(mu + 1.96 * sigma),
            'mu': mu,
            'sigma': sigma
        }

    def get_summary_stats(self, n_samples=10000, quarter=None):
        """Compute summary statistics via Monte Carlo (in MILLIONS USD)."""
        samples = self.sample(n_samples, quarter)
        return {
            'mean_M': np.mean(samples),
            'std_M': np.std(samples),
            'median_M': np.median(samples),
            'p5_M': np.percentile(samples, 5),
            'p95_M': np.percentile(samples, 95)
        }


# ============================================================================
# VALIDATION
# ============================================================================
print("=" * 70)
print("SHOCK DISTRIBUTION VALIDATION")
print("=" * 70)

shock_dist = ShockDistribution(distribution_type='lognormal', seed=42)

print("\n" + "-" * 70)
print("Time-Varying Stress Scaling (√stress on location):")
print("-" * 70)
shock_dist_tv = ShockDistribution(distribution_type='time_varying', seed=42)

quarters_sorted = sorted(ShockDistribution.STRESS_FACTORS.keys())
print(f"\n{'Quarter':<12} {'Stress':>8} {'√Stress':>8} {'Median':>10} {'Mean':>10}")
print("-" * 50)
for quarter in quarters_sorted:
    stress = ShockDistribution.STRESS_FACTORS[quarter]
    stats = shock_dist_tv.get_implied_stats(quarter)
    print(f"{quarter:<12} {stress:>8.2f} {np.sqrt(stress):>8.2f} ${stats['median_B']:>7.0f}B ${stats['mean_B']:>7.0f}B")

print("\n" + "=" * 70)
print("✓ Validation checks completed.")
print("=" * 70)

In [4]:
# @title Simulation Main (Stochastic Shocks) - PUBLICATION HARDENED v3.2 FINAL
import numpy as np
import hashlib
from tqdm.auto import tqdm
from itertools import permutations
from time import perf_counter as ptime

# ============================================================================
# GLOBAL CONSTANTS (Centrally controlled, auditable)
# ============================================================================

# Precompute permutations once
PMT = list(permutations([0, 1, 2, 3]))

# Numerical tolerance
EPS = 1e-9

# Failure rule (centrally defined)
FAILURE_RULE = "external_creditors_hit"  # TR[2] > EPS

# Baseline shock (self-documenting units)
BASELINE_SHOCK_B = 442.0                  # Blinder (2014) reference in BILLIONS
BASELINE_SHOCK_M = BASELINE_SHOCK_B * 1000  # Converted to MILLIONS for simulation


def is_failed(TR, eps=EPS):
    """
    Determine if a bank has failed.

    Definition: A bank fails when losses reach external creditors,
    i.e., losses exceeded both equity and interbank absorption capacity.
    """
    return TR[2] > eps


# ============================================================================
# STREAMING STATISTICS CLASSES
# ============================================================================

class WelfordAccumulator:
    """Welford (1962) online algorithm for streaming mean/variance."""
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2

    def variance(self):
        return self.M2 / self.n if self.n > 1 else 0.0

    def std(self):
        return np.sqrt(self.variance())


class ReservoirSampler:
    """Vitter (1985) Algorithm R for uniform sampling from stream."""
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.reservoir = []
        self.n = 0

    def update(self, x, rng):
        self.n += 1
        if len(self.reservoir) < self.capacity:
            self.reservoir.append(x)
        else:
            j = rng.integers(0, self.n)
            if j < self.capacity:
                self.reservoir[j] = x

    def percentile(self, q):
        return np.percentile(self.reservoir, q) if self.reservoir else np.nan

    def cvar(self, q):
        if not self.reservoir:
            return np.nan
        threshold = self.percentile(q)
        tail = [x for x in self.reservoir if x >= threshold]
        return np.mean(tail) if tail else np.nan


# ============================================================================
# FILL FUNCTION (Waterfall Loss Absorption)
# ============================================================================

def fill(AR, AC, qt):
    """
    Waterfall loss absorption mechanism.
    Distributes loss `qt` across compartments: Equity → Interbank → External.
    """
    if qt + AR[0] > AC[0]:
        overflow = qt + AR[0] - AC[0]
        AR[0] = AC[0]

        if overflow + AR[1] > AC[1]:
            overflow2 = overflow + AR[1] - AC[1]
            AR[1] = AC[1]
            AR[2] += overflow2
        else:
            AR[1] += overflow
    else:
        AR[0] += qt

    return AR


# ============================================================================
# DETERMINISTIC PER-QUARTER SEEDING (Process-invariant via hashlib)
# ============================================================================

def derive_quarter_seed(base_seed, quarter_name):
    """
    Derive deterministic per-quarter seed from base seed and quarter name.

    Uses SHA-256 for process-invariant hashing (unlike Python's hash(),
    which is salted per interpreter process).

    Ensures:
    - Results are invariant to quarter execution order
    - Results are invariant across different Python processes/machines
    - Safe for parallelization
    - Reproducible given same base_seed + quarter_name
    """
    h = hashlib.sha256(quarter_name.encode("utf-8")).digest()
    q_hash = int.from_bytes(h[:4], "little")  # 32-bit deterministic
    return (base_seed + q_hash) % (2**32)


# ============================================================================
# DETERMINISTIC BASELINE COMPUTATION
# ============================================================================

def compute_deterministic_baseline(MC, shock_value, n_network_samples=50000, seed=999):
    """Compute baseline SR using constant shock."""
    rng = np.random.default_rng(seed)

    sr_accum = WelfordAccumulator()
    ph_counts = [0, 0, 0, 0, 0]

    for _ in range(n_network_samples):
        c = rng.integers(0, len(PMT))
        T1C, T2C = MC[PMT[c][0]], MC[PMT[c][1]]
        T3C, T4C = MC[PMT[c][2]], MC[PMT[c][3]]

        i = rng.integers(1, 26)
        j = rng.integers(1, 26 - i) if i < 25 else 1
        l = rng.integers(1, 26)
        n_param = rng.integers(1, 26)

        T1R, T2R, T3R, T4R = [0,0,0], [0,0,0], [0,0,0], [0,0,0]

        T1R = fill(T1R, T1C, shock_value)
        k = 100 - i - j
        T2R = fill(T2R, T2C, T1R[1] * (i/100))
        T3R = fill(T3R, T3C, T1R[1] * (j/100))
        T4R = fill(T4R, T4C, T1R[1] * (k/100))
        T1R[1] = 0

        m = 100 - l
        T3R = fill(T3R, T3C, T2R[1] * (l/100))
        T4R = fill(T4R, T4C, T2R[1] * (m/100))
        T2R[1] = 0

        T4R = fill(T4R, T4C, T3R[1] * (n_param/100))
        T3R[1] -= T3R[1] * (n_param/100)

        sr = T1R[0]+T2R[0]+T3R[0]+T4R[0] + T1R[2]+T2R[2]+T3R[2]+T4R[2]
        sr_accum.update(sr)

        n_failures = sum(1 for TR in [T1R,T2R,T3R,T4R] if is_failed(TR))
        ph_counts[n_failures] += 1

    return {
        'mean_SR': sr_accum.mean,
        'std_SR': sr_accum.std(),
        'se_SR': sr_accum.std() / np.sqrt(n_network_samples),
        'shock_value': shock_value,
        'PH_probs': [c / n_network_samples for c in ph_counts],
    }


# ============================================================================
# MAIN STOCHASTIC SIMULATION (v3.2 FINAL)
# ============================================================================

def run_stochastic_simulation(MC, quarter_name, shock_dist,
                               n_shock_samples=500,
                               n_network_samples=50000,
                               shock_seed=42,
                               network_seed=12345,
                               reservoir_capacity=100000,
                               show_progress=True,
                               strict_identity_check=False):
    """
    Publication-hardened stochastic Monte Carlo simulation.

    v3.2 FINAL:
    - Process-invariant seeding via hashlib (not Python hash())
    - Self-documenting baseline shock constant
    - Corrected interval labeling (distributional, not CI)

    Failure Definition
    ------------------
    A bank "fails" when losses reach external creditors (TR[2] > EPS).
    Controlled by global FAILURE_RULE and is_failed() function.

    Reproducibility
    ---------------
    Seeds are derived per-quarter using SHA-256 hash of quarter_name.
    Results are invariant across:
    - Different Python processes
    - Different machines
    - Different execution orders
    - Parallel runs

    Uncertainty Quantification
    --------------------------
    - se_mean_SR_shock: Standard error of mean SR (correct for inference)
    - shock_cond_mean_p2.5/p97.5: 95% interval over shock-conditional means
      (distributional interval, NOT a classical confidence interval)
    """

    A = ptime()
    total_scenarios = n_shock_samples * n_network_samples

    # ========================================================================
    # PER-QUARTER DETERMINISTIC SEEDING (Process-invariant)
    # ========================================================================
    shock_seed_q = derive_quarter_seed(shock_seed, quarter_name)
    network_seed_q = derive_quarter_seed(network_seed, quarter_name)

    rng_net = np.random.default_rng(network_seed_q)

    print(f"{'='*70}")
    print(f"STOCHASTIC SIMULATION: {quarter_name}")
    print(f"{'='*70}")
    print(f"  Shock samples: {n_shock_samples}")
    print(f"  Network samples per shock: {n_network_samples:,}")
    print(f"  Total scenarios: {total_scenarios:,}")
    print(f"  Seeds (SHA-256 derived): shock={shock_seed_q}, network={network_seed_q}")
    print(f"  Failure rule: {FAILURE_RULE}")

    # ========================================================================
    # GENERATE SHOCK SAMPLES
    # ========================================================================
    shock_dist_local = shock_dist.__class__(
        distribution_type=shock_dist.dist_type,
        seed=shock_seed_q
    )
    shock_samples = shock_dist_local.sample(n_shock_samples, quarter=quarter_name)

    print(f"\nShock distribution ({shock_dist.__class__.__name__}):")
    print(f"  Mean: ${np.mean(shock_samples):,.0f}M (${np.mean(shock_samples)/1000:.1f}B)")
    print(f"  Median: ${np.median(shock_samples):,.0f}M")

    # ========================================================================
    # STREAMING ACCUMULATORS
    # ========================================================================
    sr_total_accum = WelfordAccumulator()
    e_total_accum = WelfordAccumulator()
    h_total_accum = WelfordAccumulator()
    sr_reservoir = ReservoirSampler(capacity=reservoir_capacity)

    PH_total = [0, 0, 0, 0, 0]
    shock_means = []
    shock_variances = []
    shock_p_ge_1 = []

    # ========================================================================
    # MAIN SIMULATION LOOP
    # ========================================================================
    if show_progress:
        pbar = tqdm(enumerate(shock_samples), total=n_shock_samples,
                   desc=f"Stochastic MC: {quarter_name}")
    else:
        pbar = enumerate(shock_samples)

    for shock_idx, s in pbar:
        sr_shock_accum = WelfordAccumulator()
        ph_shock = [0, 0, 0, 0, 0]

        for _ in range(n_network_samples):
            c = rng_net.integers(0, len(PMT))
            T1C, T2C = MC[PMT[c][0]], MC[PMT[c][1]]
            T3C, T4C = MC[PMT[c][2]], MC[PMT[c][3]]

            i = rng_net.integers(1, 26)
            j = rng_net.integers(1, 26 - i) if i < 25 else 1
            l = rng_net.integers(1, 26)
            n_param = rng_net.integers(1, 26)

            T1R, T2R, T3R, T4R = [0,0,0], [0,0,0], [0,0,0], [0,0,0]

            T1R = fill(T1R, T1C, s)
            k = 100 - i - j
            T2R = fill(T2R, T2C, T1R[1] * (i/100))
            T3R = fill(T3R, T3C, T1R[1] * (j/100))
            T4R = fill(T4R, T4C, T1R[1] * (k/100))
            T1R[1] = 0

            m = 100 - l
            T3R = fill(T3R, T3C, T2R[1] * (l/100))
            T4R = fill(T4R, T4C, T2R[1] * (m/100))
            T2R[1] = 0

            T4R = fill(T4R, T4C, T3R[1] * (n_param/100))
            T3R[1] -= T3R[1] * (n_param/100)

            e_loss = T1R[0] + T2R[0] + T3R[0] + T4R[0]
            h_loss = T1R[2] + T2R[2] + T3R[2] + T4R[2]
            sr = e_loss + h_loss

            sr_total_accum.update(sr)
            e_total_accum.update(e_loss)
            h_total_accum.update(h_loss)
            sr_shock_accum.update(sr)
            sr_reservoir.update(sr, rng_net)

            n_failures = sum(1 for TR in [T1R,T2R,T3R,T4R] if is_failed(TR))
            PH_total[n_failures] += 1
            ph_shock[n_failures] += 1

        shock_means.append(sr_shock_accum.mean)
        shock_variances.append(sr_shock_accum.variance())
        shock_p_ge_1.append(1 - ph_shock[0] / n_network_samples)

        if show_progress:
            pbar.set_postfix({
                'shock': f'${s/1000:.0f}B',
                'mean_SR': f'${sr_shock_accum.mean/1000:.1f}B'
            })

    # ========================================================================
    # BASELINE COMPUTATION (Self-documenting constant)
    # ========================================================================
    print("\nComputing deterministic baseline...")
    baseline = compute_deterministic_baseline(
        MC, BASELINE_SHOCK_M, n_network_samples=n_network_samples, seed=network_seed_q
    )
    print(f"  Baseline (s=${BASELINE_SHOCK_B:.0f}B): "
          f"${baseline['mean_SR']/1000:.1f}B ± ${baseline['se_SR']/1000:.2f}B (SE)")

    # ========================================================================
    # STANDARD ERRORS & INTERVALS
    # ========================================================================
    # Shock-level SE: correct for inference about mean SR
    se_mean_SR_shock = np.std(shock_means, ddof=1) / np.sqrt(n_shock_samples)

    # Scenario-level SE: less meaningful (hierarchical dependence)
    se_SR_scenario = sr_total_accum.std() / np.sqrt(total_scenarios)

    # Distributional interval over shock-conditional means
    # NOTE: This is NOT a classical CI for the estimator; it captures
    # the range of E[SR|s] across shock draws (shock uncertainty)
    shock_cond_mean_p2_5 = np.percentile(shock_means, 2.5)
    shock_cond_mean_p97_5 = np.percentile(shock_means, 97.5)

    # ========================================================================
    # VARIANCE DECOMPOSITION
    # ========================================================================
    var_total = sr_total_accum.variance()
    var_between = np.var(shock_means, ddof=0)
    var_within = np.mean(shock_variances)

    identity_sum = var_within + var_between
    identity_gap = var_total - identity_sum
    identity_gap_pct = abs(identity_gap) / var_total * 100 if var_total > 0 else 0

    print(f"\nVariance Decomposition:")
    print(f"  Total:   {var_total:,.0f}")
    print(f"  Between: {var_between:,.0f} ({var_between/var_total*100:.1f}%)")
    print(f"  Within:  {var_within:,.0f} ({var_within/var_total*100:.1f}%)")
    print(f"  Gap:     {identity_gap_pct:.3f}%", end=" ")

    if identity_gap_pct > 1.0:
        print("⚠️")
        if strict_identity_check:
            raise RuntimeError(
                f"Law of total variance check failed: gap={identity_gap_pct:.3f}% > 1%."
            )
    else:
        print("✓")

    # ========================================================================
    # COMPILE RESULTS
    # ========================================================================
    PH_probs = [c / total_scenarios for c in PH_total]
    execution_time = ptime() - A

    results = {
        'quarter': quarter_name,
        'n_shock_samples': n_shock_samples,
        'n_network_samples': n_network_samples,
        'n_total_scenarios': total_scenarios,
        'execution_time_sec': execution_time,
        'failure_rule': FAILURE_RULE,

        # Seeds (for reproducibility)
        'shock_seed_base': shock_seed,
        'network_seed_base': network_seed,
        'shock_seed_quarter': shock_seed_q,
        'network_seed_quarter': network_seed_q,

        # Shock distribution
        'shock_mean': np.mean(shock_samples),
        'shock_std': np.std(shock_samples),
        'shock_median': np.median(shock_samples),
        'shock_samples': shock_samples,

        # Systemic risk
        'mean_SR': sr_total_accum.mean,
        'std_SR': sr_total_accum.std(),
        'mean_E': e_total_accum.mean,
        'mean_H': h_total_accum.mean,

        # Standard errors
        'se_mean_SR_shock': se_mean_SR_shock,
        'se_SR_scenario': se_SR_scenario,

        # Tail risk
        'VaR95_SR': sr_reservoir.percentile(95),
        'VaR99_SR': sr_reservoir.percentile(99),
        'CVaR95_SR': sr_reservoir.cvar(95),

        # Distributional interval over shock-conditional means
        # (captures shock uncertainty, NOT a classical CI)
        'shock_cond_mean_p2.5': shock_cond_mean_p2_5,
        'shock_cond_mean_p97.5': shock_cond_mean_p97_5,

        # Failure probabilities
        'PH_probs': PH_probs,
        'P_at_least_1': 1 - PH_probs[0],
        'P_at_least_1_interval_lower': np.percentile(shock_p_ge_1, 2.5),
        'P_at_least_1_interval_upper': np.percentile(shock_p_ge_1, 97.5),

        # Variance decomposition
        'var_total': var_total,
        'var_between': var_between,
        'var_within': var_within,
        'var_between_pct': var_between/var_total*100 if var_total > 0 else 0,
        'identity_gap_pct': identity_gap_pct,

        # Baseline
        'baseline_shock_B': BASELINE_SHOCK_B,
        'baseline_mean_SR': baseline['mean_SR'],
        'baseline_se_SR': baseline['se_SR'],

        # Per-shock data
        'shock_means': shock_means,
        'shock_p_ge_1': shock_p_ge_1,
    }

    # ========================================================================
    # SUMMARY
    # ========================================================================
    print(f"\n{'='*70}")
    print(f"COMPLETE: {quarter_name} ({execution_time:.1f}s)")
    print(f"{'='*70}")
    print(f"Mean SR: ${results['mean_SR']/1000:.1f}B ± ${se_mean_SR_shock/1000:.2f}B (SE)")
    print(f"95% shock-conditional interval: [${shock_cond_mean_p2_5/1000:.1f}B, ${shock_cond_mean_p97_5/1000:.1f}B]")
    print(f"Baseline: ${baseline['mean_SR']/1000:.1f}B (Δ={100*(results['mean_SR']/baseline['mean_SR']-1):+.1f}%)")
    print(f"P(≥1 fail): {results['P_at_least_1']*100:.2f}%")

    return results

In [ ]:
# @title Verify hash stability (process- and machine-invariant)
import hashlib

def derive_quarter_seed(base_seed, quarter_name):
    h = hashlib.sha256(quarter_name.encode("utf-8")).digest()
    q_hash = int.from_bytes(h[:4], "little")
    return (base_seed + q_hash) % (2**32)

quarters = ['Q3 2007', 'Q4 2007', 'Q1 2008', 'Q2 2008', 'Q3 2008', 'Q4 2008']
base_seeds = [1, 42, 999]

print("Hash stability test (values must match across machines/processes):\n")

for b in base_seeds:
    print(f"Base seed = {b}")
    for q in quarters:
        print(f"  {q}: {derive_quarter_seed(b, q)}")
    print()


In [ ]:
# @title Run All Quarters (v3.2 FINAL - Process-Invariant)

# ============================================================================
# BALANCE SHEET DATA (All Quarters, Millions USD)
# Format: [[Equity, Interbank, External], ...] for [JPM, Citi, BofA, WFC]
# ============================================================================
BALANCE_SHEETS = {
    # Q3 2007 (Lines 917-920) - Note: 228556.6 rounded to 228557
    'Q3 2007': [[127113, 228557, 812850],
                [119978, 162303, 678091],
                [138510, 143979, 699222],
                [47738, 4546, 334956]],

    # Q4 2007 (Lines 828-831)
    'Q4 2007': [[113447, 274066, 826230],
                [123221, 182363, 740728],
                [146803, 141325, 805177],
                [47628, 2754, 344460]],

    # Q1 2008 (Lines 739-742)
    'Q1 2008': [[128219, 239006, 831208],
                [125627, 215590, 761626],
                [156309, 129096, 797069],
                [48159, 4171, 358144]],

    # Q2 2008 (Lines 650-653)
    'Q2 2008': [[136405, 220169, 803642],
                [133176, 193437, 722905],
                [162691, 114719, 784764],
                [47964, 4088, 339124]],

    # Q3 2008 (Lines 561-564)
    'Q3 2008': [[126062, 225409, 780343],
                [145843, 268040, 969783],
                [161039, 98747, 874051],
                [46957, 8093, 353574]],

    # Q4 2008 (Lines 470-473)
    'Q4 2008': [[144022, 184133, 774185],
                [166884, 341254, 1009277],
                [177052, 92048, 882997],
                [102316, 49433, 781402]],
}

# ============================================================================
# INITIALIZE (Class identity preserved across all quarters)
# ============================================================================
shock_dist = ShockDistribution(distribution_type='time_varying', seed=42)
all_results = {}

# Order does not matter due to SHA-256 per-quarter seeding inside v3.2
QUARTERS = ['Q3 2007', 'Q4 2007', 'Q1 2008', 'Q2 2008', 'Q3 2008', 'Q4 2008']

# Explicitly set reservoir capacity for auditability
RES_CAP = 100000

for quarter in QUARTERS:
    if quarter not in BALANCE_SHEETS:
        print(f"[WARN] Skipping {quarter}: balance sheet not defined")
        continue

    results = run_stochastic_simulation(
        MC=BALANCE_SHEETS[quarter],
        quarter_name=quarter,
        shock_dist=shock_dist,
        n_shock_samples=500,
        n_network_samples=50000,
        shock_seed=42,
        network_seed=12345,
        reservoir_capacity=RES_CAP,
        show_progress=True,
        strict_identity_check=True
    )
    all_results[quarter] = results

# ============================================================================
# SUMMARY TABLE (Console)
# ============================================================================
print("\n" + "="*78)
print("ALL QUARTERS COMPLETE - SUMMARY")
print("="*78)
print(f"{'Quarter':<10} {'Mean SR':>12} {'SE (shock)':>12} {'P(≥1 fail)':>12} {'Between Var %':>14}")
print("-"*78)

for quarter in QUARTERS:
    if quarter not in all_results:
        continue
    r = all_results[quarter]
    print(f"{quarter:<10} "
          f"${r['mean_SR']/1000:>10.1f}B "
          f"${r['se_mean_SR_shock']/1000:>10.2f}B "
          f"{r['P_at_least_1']*100:>10.2f}% "
          f"{r['var_between_pct']:>13.1f}%")

print("-"*78)
print(f"Baseline shock: ${BASELINE_SHOCK_B:.0f}B (Blinder 2014)")
print(f"Failure rule: {FAILURE_RULE}")
print(f"Reservoir capacity: {RES_CAP:,}")
print(f"Total scenarios per quarter: {500 * 50000:,}")
print("="*78)


In [7]:
# @title Generate Publication Figures (All Quarters)

import os
import numpy as np
import matplotlib.pyplot as plt

OUTDIR = "paper_outputs/figs"
os.makedirs(OUTDIR, exist_ok=True)

for quarter in QUARTERS:
    if quarter not in all_results:
        print(f"[WARN] {quarter}: missing results (skipped).")
        continue

    r = all_results[quarter]
    q_label = quarter.replace(' ', '_')

    # ========================================================================
    # FIGURE 1: SHOCK DISTRIBUTION
    # ========================================================================
    fig, ax = plt.subplots(figsize=(8, 6), dpi=600, facecolor='white')

    shock_B = np.asarray(r['shock_samples'], dtype=float) / 1000.0  # billions
    ax.hist(shock_B, bins=40, density=True,
            color='#4682B4', alpha=0.7, edgecolor='black', linewidth=1.0)

    ax.axvline(BASELINE_SHOCK_B, color='red', linestyle='--', linewidth=2.5,
               label=f'Baseline (Blinder 2014): ${BASELINE_SHOCK_B:.0f}B')
    ax.axvline(r['shock_mean']/1000.0, color='#003366', linestyle='-', linewidth=2.5,
               label=f'Mean: ${r["shock_mean"]/1000.0:.1f}B')

    ax.set_xlabel('Shock Magnitude (Billions USD)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Probability Density', fontsize=12, fontweight='bold')
    ax.set_title(f'Empirically-Calibrated Shock Distribution\n{quarter}',
                 fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc='upper right', frameon=True)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f'Fig_Shock_Distribution_{q_label}.png'),
                dpi=600, bbox_inches='tight')
    plt.close()
    print(f"✓ {quarter}: Shock distribution")

    # ========================================================================
    # FIGURE 2: SR vs SHOCK SCATTER (shock-conditional means)
    # ========================================================================
    fig, ax = plt.subplots(figsize=(10, 6), dpi=600, facecolor='white')

    shock_vals = shock_B
    sr_vals = np.asarray(r['shock_means'], dtype=float) / 1000.0  # billions

    if len(shock_vals) != len(sr_vals):
        raise RuntimeError(f"{quarter}: length mismatch shock_samples={len(shock_vals)} vs shock_means={len(sr_vals)}")

    ax.scatter(shock_vals, sr_vals, alpha=0.5, s=25,
               c='#003366', edgecolor='white', linewidth=0.3)

    # Trend line (in B vs B)
    z = np.polyfit(shock_vals, sr_vals, 1)
    x_line = np.linspace(float(np.min(shock_vals)), float(np.max(shock_vals)), 200)
    ax.plot(x_line, np.poly1d(z)(x_line), color='red', linestyle='--', linewidth=2,
            label=f'Linear fit: SR = {z[0]:.3f}×s + {z[1]:.2f}B')

    # Baseline reference point (B, B)
    ax.scatter([BASELINE_SHOCK_B], [r['baseline_mean_SR']/1000.0],
               color='red', s=150, marker='*', zorder=5, edgecolor='black',
               label=f'Baseline SR: ${r["baseline_mean_SR"]/1000.0:.1f}B')

    ax.set_xlabel('Shock Magnitude (Billions USD)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Systemic Risk (Billions USD)', fontsize=12, fontweight='bold')
    ax.set_title(f'Systemic Risk vs. Shock Magnitude\n{quarter}',
                 fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc='upper left', frameon=True)
    ax.grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f'Fig_SR_vs_Shock_{q_label}.png'),
                dpi=600, bbox_inches='tight')
    plt.close()
    print(f"✓ {quarter}: SR vs Shock scatter")

print("\n" + "="*50)
print("All figures generated.")
print(f"Saved to: {OUTDIR}")


✓ Q3 2007: Shock distribution
✓ Q3 2007: SR vs Shock scatter
✓ Q4 2007: Shock distribution
✓ Q4 2007: SR vs Shock scatter
✓ Q1 2008: Shock distribution
✓ Q1 2008: SR vs Shock scatter
✓ Q2 2008: Shock distribution
✓ Q2 2008: SR vs Shock scatter
✓ Q3 2008: Shock distribution
✓ Q3 2008: SR vs Shock scatter
✓ Q4 2008: Shock distribution
✓ Q4 2008: SR vs Shock scatter

All figures generated.
Saved to: paper_outputs/figs


In [8]:
# @title SUMMARY STATISTICS & FIGURES (PUBLICATION-GRADE, v1.2 FINAL — Baseline Plot FIXED)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTDIR = "paper_outputs"
FIGDIR = f"{OUTDIR}/figs"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

QUARTER_ORDER = ['Q3 2007', 'Q4 2007', 'Q1 2008',
                 'Q2 2008', 'Q3 2008', 'Q4 2008']

P_CRIT = 0.5927  # Percolation threshold (59.27%)

REQUIRED_KEYS = {
    'shock_mean', 'shock_std', 'shock_median',
    'shock_samples', 'shock_means', 'shock_p_ge_1',
    'mean_SR', 'std_SR', 'mean_E', 'mean_H',
    'VaR95_SR', 'VaR99_SR', 'CVaR95_SR',
    'shock_cond_mean_p2.5', 'shock_cond_mean_p97.5',
    'PH_probs', 'P_at_least_1',
    'P_at_least_1_interval_lower', 'P_at_least_1_interval_upper',
    'var_total', 'var_between', 'var_within', 'var_between_pct',
    'se_mean_SR_shock',
    'baseline_mean_SR'
}

# =============================================================================
# HELPERS
# =============================================================================

def fmt_billions(x):
    return f"${x/1000:,.1f}B"

def pct(x):
    return 100.0 * x

def _assert_close(label, value, target, tol=1.0):
    if abs(value - target) > tol:
        raise RuntimeError(
            f"[INTEGRITY FAIL] {label}: {value:.3f} vs {target:.3f} "
            f"(tol={tol:.3f})"
        )

# =============================================================================
# INGEST + VALIDATION
# =============================================================================

results_list = []
skipped = []

for q in QUARTER_ORDER:
    if q not in all_results:
        skipped.append((q, "missing quarter"))
        continue

    r = all_results[q]

    # --- key validation ---
    missing = REQUIRED_KEYS - r.keys()
    if missing:
        raise RuntimeError(f"{q}: missing required keys: {sorted(missing)}")

    # --- PH_probs integrity ---
    PH = r['PH_probs']
    if len(PH) != 5:
        raise RuntimeError(f"{q}: PH_probs length != 5")
    if any(p < -1e-12 for p in PH):
        raise RuntimeError(f"{q}: negative PH_probs detected")
    _assert_close(f"{q}: sum(PH_probs)", sum(PH), 1.0, tol=1e-6)

    # --- probability interval validation ---
    pL = float(r['P_at_least_1_interval_lower'])
    pM = float(r['P_at_least_1'])
    pU = float(r['P_at_least_1_interval_upper'])
    if not (0.0 <= pL <= pM <= pU <= 1.0):
        raise RuntimeError(f"{q}: malformed P(≥1) interval [{pL:.4f}, {pU:.4f}] around {pM:.4f}")

    # --- shock array consistency ---
    if len(r['shock_samples']) != len(r['shock_means']):
        raise RuntimeError(f"{q}: shock_samples vs shock_means length mismatch")

    # --- variance identity (post-hoc audit) ---
    var_total = float(r['var_total'])
    var_between = float(r['var_between'])
    var_within = float(r['var_within'])

    if var_total > 0:
        gap_pct = abs(var_total - (var_between + var_within)) / var_total * 100.0
        _assert_close(f"{q}: variance identity gap (%)", gap_pct, 0.0, tol=1.0)

    results_list.append({
        'Quarter': q,

        # Shock stats
        'Shock_mean_M': r['shock_mean'],
        'Shock_std_M': r['shock_std'],

        # Systemic risk
        'Mean_SR_M': r['mean_SR'],
        'Std_SR_M': r['std_SR'],
        'Mean_E_M': r['mean_E'],
        'Mean_H_M': r['mean_H'],

        # Tail risk
        'VaR95_SR_M': r['VaR95_SR'],
        'VaR99_SR_M': r['VaR99_SR'],
        'CVaR95_SR_M': r['CVaR95_SR'],

        # Distributional intervals
        'SR_int_lo_M': r['shock_cond_mean_p2.5'],
        'SR_int_hi_M': r['shock_cond_mean_p97.5'],

        # Failures
        'P0': PH[0],
        'P1': PH[1],
        'P2': PH[2],
        'P3': PH[3],
        'P4': PH[4],
        'P_ge_1': pM,
        'P_ge_1_lo': pL,
        'P_ge_1_hi': pU,
        'E_failures': sum(i * PH[i] for i in range(5)),

        # Variance
        'Var_total_M2': var_total,
        'Var_between_M2': var_between,
        'Var_within_M2': var_within,
        'Var_between_pct': r['var_between_pct'],

        # SE
        'SE_mean_SR_M': r['se_mean_SR_shock'],

        # Baseline
        'Baseline_SR_M': r['baseline_mean_SR'],

        # Raw series (kept out of CSV)
        '_shock_samples': r['shock_samples'],
        '_shock_means': r['shock_means'],
        '_shock_p_ge_1': r['shock_p_ge_1'],
    })

print(f"Processed {len(results_list)} quarters.")
if skipped:
    print("Skipped:", skipped)

# --- ordering integrity check ---
if len(results_list) == 0:
    raise RuntimeError("No quarters processed. Check all_results and QUARTER_ORDER.")
if len(results_list) < len(QUARTER_ORDER):
    print(f"[WARN] Only {len(results_list)}/{len(QUARTER_ORDER)} quarters processed.")

# =============================================================================
# EXPORT TABLE
# =============================================================================

df_export = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith('_')}
    for r in results_list
])
df_export.to_csv(f"{OUTDIR}/stochastic_results_summary.csv", index=False)
print(f"✓ Exported: {OUTDIR}/stochastic_results_summary.csv")

# =============================================================================
# FIGURE 1 — TEMPORAL SR WITH SHOCK UNCERTAINTY (Baseline series FIXED)
# =============================================================================

quarters = [r['Quarter'] for r in results_list]
x = np.arange(len(quarters))

mean_SR_B = [r['Mean_SR_M']/1000 for r in results_list]
lo_B = [r['SR_int_lo_M']/1000 for r in results_list]
hi_B = [r['SR_int_hi_M']/1000 for r in results_list]

# FIX: baseline is quarter-specific (computed per quarter); plot as a series (not a constant line)
baseline_series_B = [r['Baseline_SR_M']/1000 for r in results_list]

fig, ax = plt.subplots(figsize=(12,7), dpi=600)
ax.fill_between(x, lo_B, hi_B, alpha=0.3, label='Shock-conditional interval')
ax.plot(x, mean_SR_B, marker='o', linewidth=3, label='Mean SR')
ax.plot(x, baseline_series_B, linestyle=':', color='gray', linewidth=2,
        label='Deterministic baseline (s=$442B)')

ax.set_xticks(x)
ax.set_xticklabels(quarters, rotation=45, ha='right')
ax.set_ylabel("Systemic Risk (B USD)", fontweight='bold')
ax.set_xlabel("Quarter", fontweight='bold')
ax.set_title("Temporal Evolution of Systemic Risk (Q3 2007–Q4 2008)", fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/Fig_Temporal_Evolution.png", bbox_inches="tight")
plt.savefig(f"{FIGDIR}/Fig_Temporal_Evolution.tiff", bbox_inches="tight")
plt.savefig(f"{FIGDIR}/Fig_Temporal_Evolution.pdf", bbox_inches="tight")
plt.close()
print("✓ Fig_Temporal_Evolution")

# =============================================================================
# FIGURE 2 — FAILURE PROBABILITY VS PERCOLATION THRESHOLD
# =============================================================================

p_ge_1 = [pct(r['P_ge_1']) for r in results_list]
p_lo = [pct(r['P_ge_1_lo']) for r in results_list]
p_hi = [pct(r['P_ge_1_hi']) for r in results_list]

fig, ax = plt.subplots(figsize=(12,7), dpi=600)
ax.fill_between(x, p_lo, p_hi, alpha=0.3)
ax.plot(x, p_ge_1, marker='s', linewidth=3, label='P(≥1 failure)')
ax.axhline(pct(P_CRIT), linestyle='--', color='green', linewidth=2,
           label=f'Critical threshold ({pct(P_CRIT):.2f}%)')
ax.set_xticks(x)
ax.set_xticklabels(quarters, rotation=45, ha='right')
ax.set_ylabel("Probability (%)", fontweight='bold')
ax.set_xlabel("Quarter", fontweight='bold')
ax.set_title("Failure Probability vs Percolation Threshold", fontweight='bold')
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/Fig_Failure_Probability.png", bbox_inches="tight")
plt.savefig(f"{FIGDIR}/Fig_Failure_Probability.tiff", bbox_inches="tight")
plt.savefig(f"{FIGDIR}/Fig_Failure_Probability.pdf", bbox_inches="tight")
plt.close()
print("✓ Fig_Failure_Probability")

# =============================================================================
# FIGURE 3 — VARIANCE DECOMPOSITION
# =============================================================================

between_raw = np.array([
    r['Var_between_M2']/r['Var_total_M2']*100 if r['Var_total_M2'] > 0 else 0
    for r in results_list
])
between = np.clip(between_raw, 0, 100)
within = np.clip(100 - between, 0, 100)

fig, ax = plt.subplots(figsize=(10,6), dpi=600)
ax.bar(x-0.2, between, 0.4, label='Shock uncertainty', edgecolor='black')
ax.bar(x+0.2, within, 0.4, label='Network uncertainty', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(quarters, rotation=45, ha='right')
ax.set_ylabel("% of total variance", fontweight='bold')
ax.set_xlabel("Quarter", fontweight='bold')
ax.set_title("Variance Decomposition: Shock vs Network Uncertainty", fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/Fig_Variance_Decomposition.png", bbox_inches="tight")
plt.savefig(f"{FIGDIR}/Fig_Variance_Decomposition.tiff", bbox_inches="tight")
plt.savefig(f"{FIGDIR}/Fig_Variance_Decomposition.pdf", bbox_inches="tight")
plt.close()
print("✓ Fig_Variance_Decomposition")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*70)
print("✓✓✓ PUBLICATION-GRADE STATISTICS & FIGURES COMPLETE ✓✓✓")
print("="*70)
print(f"Outputs in: {OUTDIR}/")
print(f"  - stochastic_results_summary.csv ({len(df_export)} rows)")
print(f"  - figs/ (3 figures × 3 formats: PNG, TIFF, PDF)")
print("="*70)


Processed 6 quarters.
✓ Exported: paper_outputs/stochastic_results_summary.csv
✓ Fig_Temporal_Evolution
✓ Fig_Failure_Probability
✓ Fig_Variance_Decomposition

✓✓✓ PUBLICATION-GRADE STATISTICS & FIGURES COMPLETE ✓✓✓
Outputs in: paper_outputs/
  - stochastic_results_summary.csv (6 rows)
  - figs/ (3 figures × 3 formats: PNG, TIFF, PDF)


In [9]:
# @title FLOW NETWORK vs SRISK COMPARISON (PUBLICATION-GRADE, v1.3 FINAL — PATCHED)
"""
Figure Generation: Flow Network vs SRISK Comparison
For MDPI International Journal of Financial Studies
Section: External Validation

Compares stochastic Flow Network systemic risk (SR) against NYU V-Lab SRISK estimates.

IMPORTANT SEMANTICS (v3.2 FINAL):
- Flow interval [shock_cond_mean_p2.5, shock_cond_mean_p97.5] is a distributional interval over
  shock-conditional means E[SR | s] across shock draws (NOT a classical CI for the grand mean).
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTDIR = "paper_outputs"
FIGDIR = f"{OUTDIR}/figs"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

QUARTER_ORDER = ['Q3 2007', 'Q4 2007', 'Q1 2008',
                 'Q2 2008', 'Q3 2008', 'Q4 2008']

# Keys required from v3.2 FINAL simulation output
REQUIRED_KEYS = {
    'mean_SR',
    'shock_cond_mean_p2.5',
    'shock_cond_mean_p97.5',
    'shock_means',  # used for optional soft check
}

# =============================================================================
# SRISK DATA (4 banks: Citi, JPM, BofA, WFC)
# =============================================================================
# Source: NYU Stern V-Lab (https://vlab.stern.nyu.edu/)
# Values: End-of-month SRISK for final month of each quarter (billions USD)
#         Sep 2007, Dec 2007, Mar 2008, Jun 2008, Sep 2008, Dec 2008
# Note: Negative institution-level SRISK (capital surplus) included in aggregation
#
# Institution-level breakdown (millions USD):
# -------------------------------------------------------------------------
# Institution          | Q3 2007  | Q4 2007  | Q1 2008  | Q2 2008  | Q3 2008  | Q4 2008
# -------------------------------------------------------------------------
# Citigroup Inc        |  58,864  |  98,269  | 129,667  | 122,313  | 108,150  | 133,490
# JPMorgan Chase       |  32,566  |  41,802  |  49,355  |  73,417  |  99,492  | 117,405
# Bank of America      | -33,870  |  12,057  |  23,420  |  62,979  |  63,325  | 102,211
# Wells Fargo          | -37,970  | -23,423  | -18,275  |  -5,817  | -29,789  |  39,985
# -------------------------------------------------------------------------
# Aggregate            |  19,591  | 128,705  | 184,167  | 252,892  | 241,178  | 393,091
# -------------------------------------------------------------------------

SRISK_4BANKS_B = {
    'Q3 2007': 19.6,    # Aggregate: $19,590.70M; BofA & WFC negative
    'Q4 2007': 128.7,   # Aggregate: $128,704.50M; WFC still negative
    'Q1 2008': 184.2,   # Aggregate: $184,166.90M; WFC still negative
    'Q2 2008': 252.9,   # Aggregate: $252,891.70M; WFC still negative
    'Q3 2008': 241.2,   # Aggregate: $241,177.60M; WFC negative again
    'Q4 2008': 393.1,   # Aggregate: $393,091.20M; All 4 positive
}

# Interval/consistency thresholds
REL_GAP_WARN = 0.02   # warn if mean_SR differs from mean(shock_means) by >2%
EPS_POS = 1e-12       # positivity threshold for SRISK in ratio plots

# =============================================================================
# HELPERS
# =============================================================================

def _fail(msg: str):
    raise RuntimeError(msg)

def _warn(msg: str):
    print(f"[WARN] {msg}")

def _pearson_corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 2 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def _rankdata(a: np.ndarray) -> np.ndarray:
    """Simple rankdata implementation (average ranks for ties). Avoids scipy dependency."""
    a = np.asarray(a)
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(a) + 1, dtype=float)
    # average ranks for ties
    sorted_a = a[order]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sorted_a[j + 1] == sorted_a[i]:
            j += 1
        if j > i:
            avg = (i + 1 + j + 1) / 2.0
            ranks[order[i:j + 1]] = avg
        i = j + 1
    return ranks

def _spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 2:
        return np.nan
    return _pearson_corr(_rankdata(x), _rankdata(y))

# =============================================================================
# INGEST + VALIDATION
# =============================================================================

if 'all_results' not in globals():
    _fail("Missing global: all_results (run simulation cells first)")

flow_data = []
skipped = []

for q in QUARTER_ORDER:
    if q not in all_results:
        skipped.append((q, "missing quarter in all_results"))
        continue

    if q not in SRISK_4BANKS_B:
        skipped.append((q, "missing quarter in SRISK_4BANKS_B"))
        continue

    r = all_results[q]

    missing = REQUIRED_KEYS - r.keys()
    if missing:
        _fail(f"{q}: missing required keys: {sorted(missing)}")

    # Pull Flow quantities (v3.2: all in MILLIONS USD)
    lo = float(r['shock_cond_mean_p2.5'])
    mu = float(r['mean_SR'])
    hi = float(r['shock_cond_mean_p97.5'])

    # Interval sanity: only require lo <= hi
    if not (lo <= hi):
        _fail(f"{q}: malformed SR interval (lo>hi): lo={lo:.0f}, hi={hi:.0f}")

    # Optional soft check: grand mean vs average conditional mean
    mu_cond = float(np.mean(r['shock_means']))
    rel_gap = abs(mu - mu_cond) / max(abs(mu), 1.0)
    if rel_gap > REL_GAP_WARN:
        _warn(f"{q}: mean_SR differs from mean(shock_means) by {rel_gap*100:.2f}%")

    # SRISK positivity for ratio plot
    srisk_q = float(SRISK_4BANKS_B[q])
    if srisk_q <= EPS_POS:
        _fail(f"{q}: SRISK_B must be > 0 for ratio plot (got {srisk_q})")

    flow_data.append({
        'Quarter': q,
        'Flow_SR_mean_B': mu / 1000.0,
        'Flow_SR_lo_B': lo / 1000.0,
        'Flow_SR_hi_B': hi / 1000.0,
        'SRISK_B': srisk_q,
    })

print(f"Processed {len(flow_data)} quarters for comparison.")
if skipped:
    print("Skipped:", skipped)

if len(flow_data) == 0:
    _fail("No quarters processed. Check all_results + SRISK_4BANKS_B coverage.")

if len(flow_data) != len(QUARTER_ORDER):
    _warn(f"Comparison uses subset of quarters: {[d['Quarter'] for d in flow_data]}")

# =============================================================================
# COMPUTE DERIVED QUANTITIES
# =============================================================================

quarters = [d['Quarter'] for d in flow_data]
x = np.arange(len(quarters))

flow_mu = np.array([d['Flow_SR_mean_B'] for d in flow_data], dtype=float)
flow_lo = np.array([d['Flow_SR_lo_B'] for d in flow_data], dtype=float)
flow_hi = np.array([d['Flow_SR_hi_B'] for d in flow_data], dtype=float)
srisk = np.array([d['SRISK_B'] for d in flow_data], dtype=float)

# Ratio and its uncertainty
ratio_mu = flow_mu / srisk
ratio_lo = flow_lo / srisk
ratio_hi = flow_hi / srisk

# Absolute and percent deltas
delta_abs_B = flow_mu - srisk
delta_pct = (ratio_mu - 1.0) * 100.0

# Parity exclusion (two-sided) + direction
excludes_parity = (ratio_hi < 1.0) | (ratio_lo > 1.0)
parity_dir = np.where(ratio_lo > 1.0, "above",
             np.where(ratio_hi < 1.0, "below", "contains"))

# Maximum divergence indices (PATCH: used later; replaces hard-coded i0=0 narrative)
i_max_gap = int(np.argmax(np.abs(delta_abs_B)))   # max absolute gap in B
i_max_ratio = int(np.argmax(ratio_mu))            # max ratio

# Correlations
pearson_r = _pearson_corr(flow_mu, srisk)
spearman_r = _spearman_corr(flow_mu, srisk)

# =============================================================================
# FIGURE: DUAL-PANEL COMPARISON
# =============================================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 9), dpi=600,
    gridspec_kw={'height_ratios': [2, 1]}
)

# --- Panel 1: Absolute Values ---
ax1.fill_between(
    x, flow_lo, flow_hi,
    alpha=0.25, color='steelblue',
    label='Shock-conditional interval (2.5–97.5%)'
)
ax1.plot(
    x, flow_mu, marker='o', markersize=10, linewidth=3,
    color='steelblue', markeredgecolor='black', markeredgewidth=1.5,
    label='Flow Network SR', zorder=5
)
ax1.errorbar(
    x, flow_mu, yerr=[flow_mu - flow_lo, flow_hi - flow_mu],
    fmt='none', ecolor='steelblue', elinewidth=2, capsize=5, alpha=0.7
)

ax1.plot(
    x, srisk, marker='s', markersize=10, linewidth=3,
    color='coral', markeredgecolor='black', markeredgewidth=1.5,
    label='SRISK (4 banks)', zorder=5
)

# Divergence shading
ax1.fill_between(
    x, flow_mu, srisk, where=(flow_mu > srisk),
    alpha=0.12, color='green', label='Flow Network > SRISK'
)
ax1.fill_between(
    x, flow_mu, srisk, where=(flow_mu < srisk),
    alpha=0.10, color='purple', label='Flow Network < SRISK'
)

# Crisis period shading (index-based; safe for full 6-quarter set)
ax1.axvspan(3.5, 5.5, alpha=0.1, color='red', zorder=0)
ax1.text(
    4.5, max(flow_hi) * 1.05, 'Peak Crisis\n(Post-Lehman)',
    ha='center', va='bottom', fontsize=9, style='italic',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='red')
)

ax1.set_xticks(x)
ax1.set_xticklabels(quarters, rotation=45, ha='right')
ax1.set_ylabel('Systemic Risk (Billions USD)', fontweight='bold')
ax1.set_title(
    'Flow Network Model vs SRISK: Systemic Risk Evolution (2007–2008)',
    fontweight='bold', pad=12
)
ax1.set_ylim(0, max(flow_hi) * 1.2)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(loc='upper left', frameon=True, ncol=2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Panel 2: Ratio ---
ax2.fill_between(
    x, ratio_lo, ratio_hi,
    alpha=0.25, color='darkgreen',
    label='Ratio interval (from shock-conditional bounds)'
)
ax2.plot(
    x, ratio_mu, marker='D', markersize=9, linewidth=3,
    color='darkgreen', markeredgecolor='black', markeredgewidth=1.5,
    label='Flow Network / SRISK', zorder=5
)
ax2.errorbar(
    x, ratio_mu, yerr=[ratio_mu - ratio_lo, ratio_hi - ratio_mu],
    fmt='none', ecolor='darkgreen', elinewidth=2, capsize=5, alpha=0.7
)

ax2.axhline(1.0, color='black', linestyle='--', linewidth=2, label='Parity (Ratio = 1.0)')
ax2.fill_between(x, 1.0, ratio_mu, where=(ratio_mu > 1.0), alpha=0.08, color='steelblue')
ax2.fill_between(x, 1.0, ratio_mu, where=(ratio_mu < 1.0), alpha=0.08, color='purple')
ax2.axvspan(3.5, 5.5, alpha=0.1, color='red', zorder=0)

ax2.set_xticks(x)
ax2.set_xticklabels(quarters, rotation=45, ha='right')
ax2.set_xlabel('Quarter', fontweight='bold')
ax2.set_ylabel('Ratio (Flow Network / SRISK)', fontweight='bold')
ax2.set_title(
    'Convergence Pattern: Structural Model → Market Pricing at Peak Crisis',
    fontweight='bold', pad=10
)
ax2.set_ylim(0, max(ratio_hi) * 1.15)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.legend(loc='upper right', frameon=True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{FIGDIR}/Fig_FlowNetwork_vs_SRISK.png', bbox_inches='tight')
plt.savefig(f'{FIGDIR}/Fig_FlowNetwork_vs_SRISK.tiff', bbox_inches='tight')
plt.savefig(f'{FIGDIR}/Fig_FlowNetwork_vs_SRISK.pdf', bbox_inches='tight')
plt.close()
print('✓ Fig_FlowNetwork_vs_SRISK')

# =============================================================================
# EXPORT TABLE (CSV)
# =============================================================================

export_rows = []
for i, d in enumerate(flow_data):
    export_rows.append({
        'Quarter': d['Quarter'],
        'FlowNet_SR_B': d['Flow_SR_mean_B'],
        'FlowNet_lo_B': d['Flow_SR_lo_B'],
        'FlowNet_hi_B': d['Flow_SR_hi_B'],
        'SRISK_B': d['SRISK_B'],
        'Ratio': ratio_mu[i],
        'Ratio_lo': ratio_lo[i],
        'Ratio_hi': ratio_hi[i],
        'Delta_B': delta_abs_B[i],
        'Delta_pct': delta_pct[i],
        'Excludes_parity': bool(excludes_parity[i]),
        'Parity_direction': parity_dir[i],
    })

df_comparison = pd.DataFrame(export_rows)
df_comparison.to_csv(f'{OUTDIR}/flownet_vs_srisk_comparison.csv', index=False)
print(f'✓ Exported: {OUTDIR}/flownet_vs_srisk_comparison.csv')

# =============================================================================
# MARKDOWN TABLES (paper-ready)
# =============================================================================

print("\n" + "="*100)
print("TABLE 1: Institution-Level SRISK (Millions USD) — Reference Data")
print("="*100)
print("""
| Institution          | Q3 2007  | Q4 2007  | Q1 2008  | Q2 2008  | Q3 2008  | Q4 2008  |
|:---------------------|--------:|--------:|--------:|--------:|--------:|--------:|
| Citigroup Inc        |  58,864 |  98,269 | 129,667 | 122,313 | 108,150 | 133,490 |
| JPMorgan Chase       |  32,566 |  41,802 |  49,355 |  73,417 |  99,492 | 117,405 |
| Bank of America      | −33,870 |  12,057 |  23,420 |  62,979 |  63,325 | 102,211 |
| Wells Fargo          | −37,970 | −23,423 | −18,275 |  −5,817 | −29,789 |  39,985 |
| **Aggregate**        |  19,591 | 128,705 | 184,167 | 252,892 | 241,178 | 393,091 |
""")

print("\n" + "="*100)
print("TABLE 2: Flow Network vs SRISK Comparison (Billions USD)")
print("="*100)
print("\n| Quarter | Flow Net SR | Interval (2.5–97.5%) | SRISK   | Ratio  | Δ Abs.  | Δ %       | Excl. 1.0? |")
print("|:-------:|------------:|:--------------------:|--------:|-------:|--------:|----------:|:----------:|")
for i, d in enumerate(flow_data):
    int_str = f"[{d['Flow_SR_lo_B']:.0f}–{d['Flow_SR_hi_B']:.0f}]"
    excl = "YES" if excludes_parity[i] else "NO"
    print(f"| {d['Quarter']} | ${d['Flow_SR_mean_B']:10.1f} | {int_str:^20} | "
          f"${d['SRISK_B']:6.1f} | {ratio_mu[i]:6.2f} | {delta_abs_B[i]:+7.1f} | "
          f"{delta_pct[i]:+8.1f}% | {excl:^10} |")

# =============================================================================
# KEY FINDINGS (PATCHED: use computed max-divergence indices, not hard-coded)
# =============================================================================

print("\n" + "="*100)
print("KEY FINDINGS")
print("="*100)

print(f"\nAlignment diagnostics:")
print(f"  Pearson corr (levels):  {pearson_r:.3f}")
print(f"  Spearman corr (ranks):  {spearman_r:.3f}")

# PATCH: pick which definition of "maximum divergence" you want to report
# - Use i_max_gap for maximum absolute gap in billions
# - Use i_max_ratio for maximum ratio
i0 = i_max_gap

print(f"\n1. MAXIMUM DIVERGENCE (by |Δ|) ({quarters[i0]}):")
print(f"   Flow Network: ${flow_mu[i0]:.1f}B [{flow_lo[i0]:.0f}B – {flow_hi[i0]:.0f}B]")
print(f"   SRISK:        ${srisk[i0]:.1f}B")
print(f"   Ratio:        {ratio_mu[i0]:.2f}× [{ratio_lo[i0]:.1f}× – {ratio_hi[i0]:.1f}×]")
print(f"   Delta:        {delta_abs_B[i0]:+,.1f}B ({delta_pct[i0]:+,.1f}%)")
print(f"   Interpretation: Structural model identifies latent vulnerabilities before market repricing")

# Near parity (end quarter)
i_end = len(quarters) - 1
print(f"\n2. END-OF-SAMPLE ({quarters[i_end]}):")
print(f"   Flow Network: ${flow_mu[i_end]:.1f}B [{flow_lo[i_end]:.0f}B – {flow_hi[i_end]:.0f}B]")
print(f"   SRISK:        ${srisk[i_end]:.1f}B")
print(f"   Ratio:        {ratio_mu[i_end]:.2f}× [{ratio_lo[i_end]:.2f}× – {ratio_hi[i_end]:.2f}×]")
print(f"   Delta:        {delta_abs_B[i_end]:+,.1f}B ({delta_pct[i_end]:+,.1f}%)")
print(f"   Interpretation: Market pricing converges toward structural fundamentals at peak crisis")

# Progressive convergence
print(f"\n3. RATIO EVOLUTION:")
print("   " + " → ".join([f"{ratio_mu[i]:.2f}×" for i in range(len(quarters))]))

# Gap evolution (now consistent with i0 definition)
gap_initial = abs(delta_abs_B[i0])
gap_final = abs(delta_abs_B[i_end])
gap_reduction = (1 - gap_final / gap_initial) * 100 if gap_initial > 0 else np.nan
print(f"\n4. GAP EVOLUTION (relative to max-|Δ| quarter):")
print(f"   {quarters[i0]} abs gap:     ${gap_initial:.1f}B")
print(f"   {quarters[i_end]} abs gap:  ${gap_final:.1f}B")
print(f"   Reduction:            {gap_reduction:.1f}%")

# Statistical significance
n_significant = int(np.sum(excludes_parity))
n_above = int(np.sum(parity_dir == 'above'))
n_below = int(np.sum(parity_dir == 'below'))
n_contains = int(np.sum(parity_dir == 'contains'))
print(f"\n5. STATISTICAL EXCLUSION OF PARITY (two-sided):")
print(f"   Quarters with ratio interval excluding 1.0: {n_significant}/{len(quarters)}")
print(f"   Direction: above={n_above}, below={n_below}, contains={n_contains}")

# Uncertainty quantification
avg_width = float(np.mean(flow_hi - flow_lo))
avg_width_pct = avg_width / max(float(np.mean(flow_mu)), 1e-12) * 100.0
print(f"\n6. UNCERTAINTY QUANTIFICATION:")
print(f"   Average interval width: ${avg_width:.1f}B")
print(f"   As % of mean Flow SR:   {avg_width_pct:.1f}%")

# =============================================================================
# INTERPRETATION FOR MANUSCRIPT (PATCHED to reference i0-derived quarter)
# =============================================================================

print("\n" + "="*100)
print("INTERPRETATION FOR MANUSCRIPT")
print("="*100)
print(f"""
The divergence–convergence pattern validates the complementarity of deterministic
structural models and market-based risk measures:

1. MAXIMUM DIVERGENCE (by |Δ|) ({quarters[i0]}):
   - Flow Network exceeds SRISK by factor of {ratio_mu[i0]:.2f}× [{ratio_lo[i0]:.2f}×–{ratio_hi[i0]:.2f}×]
   - Even with shock uncertainty, the structural model robustly identifies
     latent vulnerabilities BEFORE market repricing (interval semantics noted above).

2. PROGRESSIVE CONVERGENCE (2007Q4 – 2008Q4):
   - Ratio path: {" → ".join([f"{ratio_mu[i]:.2f}×" for i in range(len(quarters))])}
   - Market-implied SRISK gradually incorporates balance-sheet fragilities.

3. END-OF-SAMPLE ({quarters[i_end]}):
   - Ratio = {ratio_mu[i_end]:.2f}× [{ratio_lo[i_end]:.2f}×–{ratio_hi[i_end]:.2f}×]
   - Market pricing converges toward structural fundamentals at peak crisis.
   - Gap reduction from max-|Δ| quarter: {gap_reduction:.1f}%

POLICY IMPLICATIONS:
- Flow Network provides early warning with quantified shock-conditional uncertainty bounds.
- Large ratio with interval excluding 1.0 signals meaningful divergence between structural
  vulnerability and market-implied shortfall.
- Combine both measures: Flow Network for stress testing, SRISK for real-time surveillance.
""")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*100)
print("✓✓✓ FLOW NETWORK vs SRISK COMPARISON COMPLETE ✓✓✓")
print("="*100)
print(f"Outputs in: {OUTDIR}/")
print(f"  - figs/Fig_FlowNetwork_vs_SRISK.{{png,tiff,pdf}}")
print(f"  - flownet_vs_srisk_comparison.csv")
print("="*100)


Processed 6 quarters for comparison.
✓ Fig_FlowNetwork_vs_SRISK
✓ Exported: paper_outputs/flownet_vs_srisk_comparison.csv

TABLE 1: Institution-Level SRISK (Millions USD) — Reference Data

| Institution          | Q3 2007  | Q4 2007  | Q1 2008  | Q2 2008  | Q3 2008  | Q4 2008  |
|:---------------------|--------:|--------:|--------:|--------:|--------:|--------:|
| Citigroup Inc        |  58,864 |  98,269 | 129,667 | 122,313 | 108,150 | 133,490 |
| JPMorgan Chase       |  32,566 |  41,802 |  49,355 |  73,417 |  99,492 | 117,405 |
| Bank of America      | −33,870 |  12,057 |  23,420 |  62,979 |  63,325 | 102,211 |
| Wells Fargo          | −37,970 | −23,423 | −18,275 |  −5,817 | −29,789 |  39,985 |
| **Aggregate**        |  19,591 | 128,705 | 184,167 | 252,892 | 241,178 | 393,091 |


TABLE 2: Flow Network vs SRISK Comparison (Billions USD)

| Quarter | Flow Net SR | Interval (2.5–97.5%) | SRISK   | Ratio  | Δ Abs.  | Δ %       | Excl. 1.0? |
|:-------:|------------:|:--------------------: